# Schwinger hadron challenge — Part 0: warm-up circuits

The answers for Ex 0.1–0.5 live in `schwinger_core.py` (shared with the other two notebooks). This notebook prints each answer and verifies it against exact unitaries / states.

Run from this folder (`ibm/schwinger/`) with the packages in `requirements.txt`.

In [1]:
import inspect
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector

import schwinger_core as sc
from schwinger_core import *

## The answer code (copy-paste source for the grader)

In [2]:
for f in (chiral_condensate_observables, RXYplus, RXYminus, prep_strong_coupling_vacuum, vacuum_prep_rotate_OV_3,
          prep_vacuum, wave_prep_rotate_O_22, prep_wave, trotter_step, evolve_circuits):
    print(inspect.getsource(f))

def chiral_condensate_observables(L: int) -> list[SparsePauliOp]:
    """chi_j = (-1)^j Z_j + I for j = 0 .. 2L-1 (list of 2L SparsePauliOps on 2L qubits)."""
    n = 2 * L
    obs = []
    for j in range(n):
        obs.append(SparsePauliOp.from_sparse_list([("Z", [j], (-1) ** j), ("", [], 1.0)], num_qubits=n))
    return obs

def RXYplus(theta: float) -> Gate:
    """R^{(XY)}_+(theta) = exp(-i theta/2 (XY + YX))."""
    qc = QuantumCircuit(2)
    qc.s(0); qc.h(0)
    qc.z(1); qc.h(1); qc.s(1)
    qc.cx(0, 1)
    qc.ry(theta, 0); qc.rz(theta, 1)
    qc.cx(0, 1)
    qc.h(0); qc.sdg(0)
    qc.sdg(1); qc.h(1); qc.z(1)
    return qc.to_gate(label=rf"$R^{{XY}}_{{+}}({theta:.4g})$")

def RXYminus(theta: float) -> Gate:
    """R^{(XY)}_-(theta) = exp(+i theta/2 (XY - YX))."""
    qc = QuantumCircuit(2)
    qc.s(0); qc.h(0)
    qc.z(1); qc.h(1); qc.s(1)
    qc.cx(0, 1)
    qc.ry(-theta, 0); qc.rz(theta, 1)
    qc.cx(0, 1)
    qc.h(0); qc.sdg(0)
    qc.sdg(1); qc.h(1); qc.z(1)
    return qc.to

## Ex 0.1 — chiral condensate observables

In [3]:
obs = chiral_condensate_observables(2)
print(obs[0], obs[1], sep='\n')
assert obs[1] == SparsePauliOp(['IIZI', 'IIII'], [-1, 1])

SparsePauliOp(['IIIZ', 'IIII'],
              coeffs=[1.+0.j, 1.+0.j])
SparsePauliOp(['IIZI', 'IIII'],
              coeffs=[-1.+0.j,  1.+0.j])


## Ex 0.2 — $R^{(XY)}_\pm$ against the exact exponentials (up to global phase)

In [4]:
XY = SparsePauliOp('XY').to_matrix(); YX = SparsePauliOp('YX').to_matrix()
for th in (0.731, -1.2):
    assert Operator(RXYplus(th)).equiv(Operator(expm(-1j * th / 2 * (XY + YX))))
    # the grader's ordering of XY - YX is the reverse of qiskit's label order
    assert Operator(RXYminus(th)).equiv(Operator(expm(+1j * th / 2 * (YX - XY))))
print('RXYplus / RXYminus OK')

RXYplus / RXYminus OK


## Ex 0.3 / 0.4 — vacuum and wavepacket layers vs exact (Trotterised) exponentials

In [5]:
def term(n, a, b, d, N):
    """a on qubit n, Z string, b on qubit n+d (grader's qubit ordering)."""
    s = ['I'] * N; s[n] = a; s[n + d] = b
    for k in range(n + 1, n + d):
        s[k] = 'Z'
    return SparsePauliOp(''.join(reversed(s))).to_matrix()

def pair(n, d, N):
    return 0.5 * (term(n, 'X', 'Y', d, N) - term(n, 'Y', 'X', d, N))

th = 0.37
for L in (2, 3, 4):
    N = 2 * L
    for d, layer in ((1, vacuum_prep_rotate_OV_1), (3, vacuum_prep_rotate_OV_3)):
        U = np.eye(2**N)
        for n in list(range(0, N - d, 2)) + list(range(1, N - d, 2)):
            U = expm(1j * th * (-1) ** n * pair(n, d, N)) @ U
        assert Operator(layer(QuantumCircuit(N), th, L)).equiv(Operator(U)), (L, d)
for L in (3, 4, 5):
    N = 2 * L
    O22 = pair(L - 2, 2, N) - pair(L - 1, 2, N)
    assert Operator(wave_prep_rotate_O_22(QuantumCircuit(N), th, L)).equiv(Operator(expm(1j * th * O22)))
print('O^V(1), O^V(3), O(2,2) layers OK')

O^V(1), O^V(3), O(2,2) layers OK


## Ex 0.5 — forward/backward calibration circuit returns to the initial state

In [6]:
qc_init6 = prep_wave(6, vacuum_prep_theta_OV_1, vacuum_prep_theta_OV_3, wave_prep_theta_O_11, wave_prep_theta_O_22)
qc6, qc6_mitig = evolve_circuits(qc_init6, 6, 4.0, m, g)
fid = abs(np.vdot(Statevector(qc_init6).data, Statevector(qc6_mitig).data)) ** 2
print(f'L=6, t=4 round-trip fidelity: {fid:.12f}')
assert fid > 1 - 1e-9

L=6, t=4 round-trip fidelity: 1.000000000000
